---
## Signature Detection Evaluation

Evaluates the model's ability to detect handwritten signatures using:
- **Positive samples**: SigDetectVerifyFlow dataset (documents WITH signatures)
- **Negative samples**: CORD receipts (documents WITHOUT signatures)

Metrics: Accuracy, Precision, Recall, F1

In [ ]:
# Cell 1: Install dependencies
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q accelerate bitsandbytes qwen-vl-utils[decord] datasets pillow
!pip install -q PyMuPDF opencv-python-headless matplotlib pandas

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 112.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 65.5 MB/s eta 0:00:00


In [ ]:
import os, sys

REPO_DIR = "/content/VLM-PDF-Text_Extractor"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/RahulReadd/VLM-PDF-Text_Extractor.git
os.chdir(REPO_DIR)

Cloning into 'VLM-PDF-Text_Extractor'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 71 (delta 31), reused 61 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 53.19 KiB | 2.53 MiB/s, done.
Resolving deltas: 100% (31/31), done.


In [ ]:
from app.evaluate import evaluate_signature_single, evaluate_signature_batch
from app.extract import DocumentExtractor
extractor = DocumentExtractor(model_name="qwen3-vl-2b")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

DocumentExtractor ready: qwen3-vl-2b


In [ ]:
from datasets import load_dataset
import numpy as np

N_SIG = 10  # images per class (10 positive + 10 negative = 20 total)

# Positive: documents with signatures
sig_ds = load_dataset("Mels22/SigDetectVerifyFlow", split="test")
pos_indices = np.linspace(0, len(sig_ds) - 1, N_SIG, dtype=int).tolist()
pos_images = [sig_ds[i]["document"].convert("RGB") for i in pos_indices]

# Negative: CORD receipts (no signatures)
cord = load_dataset("naver-clova-ix/cord-v2", split="test")
neg_indices = np.linspace(0, len(cord) - 1, N_SIG, dtype=int).tolist()
neg_images = [cord[i]["image"].convert("RGB") for i in neg_indices]

all_images = pos_images + neg_images
all_labels = [True] * N_SIG + [False] * N_SIG

print(f"Loaded {len(all_images)} images ({N_SIG} with signatures, {N_SIG} without)")

# Run extraction
predictions = []
for i, img in enumerate(all_images):
    result = extractor.extract(img)  # unified prompt
    predictions.append(result["parsed_json"])
    label = "+" if all_labels[i] else "-"
    print(f"  [{label}] Image {i+1}/{len(all_images)}: JSON valid={result['json_valid']}")

# Evaluate
results = evaluate_signature_batch(predictions, all_labels)

print(f"\n{'='*50}")
print(f"  SIGNATURE DETECTION RESULTS")
print(f"{'='*50}")
print(f"  Accuracy:  {results['accuracy']:.3f}")
print(f"  Precision: {results['precision']:.3f}")
print(f"  Recall:    {results['recall']:.3f}")
print(f"  F1:        {results['f1']:.3f}")
print(f"  TP={results['tp']}  FP={results['fp']}  FN={results['fn']}  TN={results['tn']}")